# Ch.3 — Evaluation Metrics for Classification

**FaceAI**: Why accuracy lies for imbalanced classes (Bald 2.5%, Mustache 4.2%).

**Goal**: Confusion matrix, precision/recall, F1, ROC-AUC, PR-AUC, multi-label metrics.

**Key insight**: 97.5% accuracy on Bald by always predicting "Not Bald" — the accuracy paradox.

In [ ]:
# ── Setup & Imports ────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score)

IMG_DIR = Path("img")
IMG_DIR.mkdir(exist_ok=True)
SAVE_KW = dict(dpi=150, bbox_inches='tight')
np.random.seed(42)
print("Imports OK")

## §0 Data — Three Attributes with Different Imbalance

In [ ]:
# ── Synthetic Attributes with Varying Imbalance ────────
def make_attr(pos_rate, name, n=5000):
    X, y = make_classification(
        n_samples=n, n_features=200, n_informative=30,
        n_redundant=20, weights=[1-pos_rate, pos_rate],
        flip_y=0.03, random_state=42
    )
    return X, y, name

datasets = {
    'Smiling (48%)': make_attr(0.48, 'Smiling'),
    'Eyeglasses (13%)': make_attr(0.13, 'Eyeglasses'),
    'Bald (2.5%)': make_attr(0.025, 'Bald'),
}

for name, (X, y, _) in datasets.items():
    print(f"{name}: {y.sum()}/{len(y)} positive ({y.mean():.1%})")

## §1 The Accuracy Paradox

In [ ]:
# ── Accuracy Paradox Demo ──────────────────────────────
X_bald, y_bald, _ = datasets['Bald (2.5%)']
X_tr, X_te, y_tr, y_te = train_test_split(X_bald, y_bald, test_size=0.2,
                                           stratify=y_bald, random_state=42)

# Model A: Always predict Not-Bald
y_pred_always_no = np.zeros_like(y_te)
acc_always_no = (y_pred_always_no == y_te).mean()
f1_always_no = f1_score(y_te, y_pred_always_no, zero_division=0)

# Model B: Logistic Regression
sc = StandardScaler()
X_tr_s, X_te_s = sc.fit_transform(X_tr), sc.transform(X_te)
lr = LogisticRegression(C=1.0, max_iter=500, class_weight='balanced', random_state=42)
lr.fit(X_tr_s, y_tr)
y_pred_lr = lr.predict(X_te_s)
acc_lr = (y_pred_lr == y_te).mean()
f1_lr = f1_score(y_te, y_pred_lr)

print("The Accuracy Paradox:")
print(f"  Always Not-Bald: accuracy={acc_always_no:.3f}, F1={f1_always_no:.3f}")
print(f"  LogReg balanced: accuracy={acc_lr:.3f}, F1={f1_lr:.3f}")
print(f"\n  → 'Dumb' model has HIGHER accuracy but ZERO F1!")

## §2 Confusion Matrices Across Attributes

In [ ]:
# ── Confusion Matrices for 3 Attributes ────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, (X, y, label)) in zip(axes, datasets.items()):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                               stratify=y, random_state=42)
    sc = StandardScaler()
    X_tr_s, X_te_s = sc.fit_transform(X_tr), sc.transform(X_te)
    lr = LogisticRegression(C=1.0, max_iter=500, random_state=42)
    lr.fit(X_tr_s, y_tr)
    
    ConfusionMatrixDisplay.from_estimator(
        lr, X_te_s, y_te, display_labels=[f'Not {label}', label],
        cmap='Blues', ax=ax
    )
    ax.set_title(name)

fig.suptitle('Confusion Matrices — Different Class Balances', fontsize=14)
fig.tight_layout()
fig.savefig(IMG_DIR / 'confusion_matrices.png', **SAVE_KW)
plt.show()

## §3 ROC Curves

In [ ]:
# ── ROC Curves for 3 Attributes ────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
colors = ['blue', 'green', 'red']

for color, (name, (X, y, _)) in zip(colors, datasets.items()):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                               stratify=y, random_state=42)
    sc = StandardScaler()
    X_tr_s, X_te_s = sc.fit_transform(X_tr), sc.transform(X_te)
    lr = LogisticRegression(C=1.0, max_iter=500, random_state=42)
    lr.fit(X_tr_s, y_tr)
    y_prob = lr.predict_proba(X_te_s)[:, 1]
    
    fpr, tpr, _ = roc_curve(y_te, y_prob)
    auc = roc_auc_score(y_te, y_prob)
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC={auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Per-Attribute')
ax.legend()
fig.savefig(IMG_DIR / 'roc_curves.png', **SAVE_KW)
plt.show()

## §4 PR Curves (Better for Imbalanced Data)

In [ ]:
# ── Precision-Recall Curves ────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))

for color, (name, (X, y, _)) in zip(colors, datasets.items()):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                               stratify=y, random_state=42)
    sc = StandardScaler()
    X_tr_s, X_te_s = sc.fit_transform(X_tr), sc.transform(X_te)
    lr = LogisticRegression(C=1.0, max_iter=500, random_state=42)
    lr.fit(X_tr_s, y_tr)
    y_prob = lr.predict_proba(X_te_s)[:, 1]
    
    prec, rec, _ = precision_recall_curve(y_te, y_prob)
    ap = average_precision_score(y_te, y_prob)
    ax.plot(rec, prec, color=color, linewidth=2, label=f'{name} (AP={ap:.3f})')

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves (better for imbalanced)')
ax.legend()
fig.savefig(IMG_DIR / 'pr_curves.png', **SAVE_KW)
plt.show()

## §5 Threshold Tuning for Bald (2.5%)

In [ ]:
# ── Threshold Tuning for Imbalanced Attribute ──────────
X_bald, y_bald, _ = datasets['Bald (2.5%)']
X_tr, X_te, y_tr, y_te = train_test_split(X_bald, y_bald, test_size=0.2,
                                           stratify=y_bald, random_state=42)
sc = StandardScaler()
X_tr_s, X_te_s = sc.fit_transform(X_tr), sc.transform(X_te)
lr = LogisticRegression(C=1.0, max_iter=500, random_state=42)
lr.fit(X_tr_s, y_tr)
y_prob = lr.predict_proba(X_te_s)[:, 1]

thresholds = np.arange(0.02, 0.8, 0.02)
f1s, precs, recs = [], [], []
for t in thresholds:
    y_t = (y_prob >= t).astype(int)
    f1s.append(f1_score(y_te, y_t, zero_division=0))
    precs.append(precision_score(y_te, y_t, zero_division=0))
    recs.append(recall_score(y_te, y_t, zero_division=0))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, f1s, 'b-', label='F1', linewidth=2)
ax.plot(thresholds, precs, 'g--', label='Precision')
ax.plot(thresholds, recs, 'r--', label='Recall')
best_t = thresholds[np.argmax(f1s)]
ax.axvline(best_t, color='blue', linestyle=':', alpha=0.7)
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5, label='Default 0.5')
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title(f'Bald Threshold Tuning — Optimal t={best_t:.2f} (default=0.5)')
ax.legend()
fig.savefig(IMG_DIR / 'bald_threshold_tuning.png', **SAVE_KW)
plt.show()

print(f"At default t=0.5: F1={f1_score(y_te, (y_prob>=0.5).astype(int), zero_division=0):.3f}")
print(f"At optimal t={best_t:.2f}: F1={max(f1s):.3f}")

## §6 Multi-Label Metrics Preview

In [ ]:
# ── Multi-Label Metrics (simulated 10 attributes) ─────
n_attrs = 10
n_samples = 1000

# Simulate predictions for 10 attributes
np.random.seed(42)
y_true_multi = np.random.binomial(1, 0.3, size=(n_samples, n_attrs))
y_pred_multi = y_true_multi.copy()
# Flip ~10% of labels to simulate errors
flip_mask = np.random.random(size=(n_samples, n_attrs)) < 0.10
y_pred_multi[flip_mask] = 1 - y_pred_multi[flip_mask]

# Hamming Loss
hamming = (y_true_multi != y_pred_multi).mean()

# Subset Accuracy (exact match across ALL attributes)
subset_acc = (y_true_multi == y_pred_multi).all(axis=1).mean()

# Per-attribute accuracy
per_attr_acc = (y_true_multi == y_pred_multi).mean(axis=0)

print(f"Hamming Loss:     {hamming:.4f} (fraction of wrong labels)")
print(f"Subset Accuracy:  {subset_acc:.4f} (exact match on ALL {n_attrs} attrs)")
print(f"Per-attr accuracy: mean={per_attr_acc.mean():.4f}, min={per_attr_acc.min():.4f}")
print(f"\n→ With {n_attrs} attributes and 10% per-label error:")
print(f"  Hamming is forgiving (~10%), subset accuracy is strict (~{subset_acc:.0%})")

## §7 Cross-Validation Stability

In [ ]:
# ── Cross-Validation ───────────────────────────────────
X_smile, y_smile, _ = datasets['Smiling (48%)']
sc = StandardScaler()
X_s = sc.fit_transform(X_smile)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lr = LogisticRegression(C=1.0, max_iter=500, random_state=42)

scores_acc = cross_val_score(lr, X_s, y_smile, cv=cv, scoring='accuracy')
scores_f1 = cross_val_score(lr, X_s, y_smile, cv=cv, scoring='f1')
scores_auc = cross_val_score(lr, X_s, y_smile, cv=cv, scoring='roc_auc')

print(f"Accuracy:  {scores_acc.mean():.3f} ± {scores_acc.std():.3f}")
print(f"F1:        {scores_f1.mean():.3f} ± {scores_f1.std():.3f}")
print(f"ROC-AUC:   {scores_auc.mean():.3f} ± {scores_auc.std():.3f}")

fig, ax = plt.subplots(figsize=(8, 4))
metrics = ['Accuracy', 'F1', 'ROC-AUC']
means = [scores_acc.mean(), scores_f1.mean(), scores_auc.mean()]
stds = [scores_acc.std(), scores_f1.std(), scores_auc.std()]
ax.bar(metrics, means, yerr=stds, capsize=5, color=['#2196F3', '#4CAF50', '#FF9800'])
ax.set_ylabel('Score')
ax.set_title('Cross-Validation Metrics — Smiling (5-fold)')
ax.set_ylim(0.7, 1.0)
fig.savefig(IMG_DIR / 'cv_stability.png', **SAVE_KW)
plt.show()

## §8 Summary

In [ ]:
# ── Chapter Summary ────────────────────────────────────
print("=" * 50)
print("Ch.3 — Evaluation Metrics Summary")
print("=" * 50)
print("\nKey Lessons:")
print("  1. Accuracy lies for imbalanced classes (Bald: 97.5% by doing nothing)")
print("  2. F1, PR-AUC are essential for rare attributes")
print("  3. Threshold tuning is critical (0.5 default is often wrong)")
print("  4. Multi-label: Hamming loss (forgiving) vs subset accuracy (strict)")
print("  5. Cross-validation gives confidence intervals")
print("\nConstraint #1 (ACCURACY): 88% VALIDATED with proper metrics")
print("Next: Ch.4 — SVM (maximum margin for better accuracy)")

## Exercises

1. **Calibration curve**: Plot the calibration curve for LogReg on Smiling. Is it well-calibrated?
2. **Matthews Correlation Coefficient**: Compute MCC for Bald. Compare with F1 — which is more informative?
3. **Multi-label heatmap**: Simulate 40 attributes, compute per-attribute F1, plot as a sorted bar chart.

In [ ]:
# Exercise 1: Calibration curve
# TODO: Use sklearn.calibration.calibration_curve, plot reliability diagram
pass

In [ ]:
# Exercise 2: Matthews Correlation Coefficient
# TODO: sklearn.metrics.matthews_corrcoef for Bald, compare with F1
pass

In [ ]:
# Exercise 3: Multi-label F1 heatmap
# TODO: Simulate 40 attributes with varying positive rates, compute per-attr F1
pass